# Task 1.3 — Math for ML Practice Set
### Statistics · Probability · Linear Algebra · Calculus — with worked solutions

Each section states an applied problem from the ML world, solves it by hand (markdown),
then verifies the result numerically with `NumPy`.
**Deliverable:** this notebook of worked solutions.

## 0. Setup

In [1]:
import numpy as np
from scipy import stats

print("numpy", np.__version__)

numpy 2.5.2


## 1. Statistics — the language of data

> **1.1** A feature `model_accuracy` over 10 models is `[0.82, 0.79, 0.91, 0.85, 0.88, 0.76, 0.84, 0.81, 0.90, 0.87]`.
> Compute the mean, median, variance and standard deviation.

**Worked solution:**
Mean = (0.82+0.79+0.91+0.85+0.88+0.76+0.84+0.81+0.90+0.87) / 10 = 8.43 / 10 = **0.843**.

Sorted: 0.76, 0.79, 0.81, 0.82, 0.84, 0.85, 0.87, 0.88, 0.90, 0.91 →
median = (0.84+0.85)/2 = **0.845**.

Variance (population) = mean of squared deviations = **0.002121**, std = √var ≈ **0.0461**.

In [2]:
scores = np.array([0.82, 0.79, 0.91, 0.85, 0.88, 0.76, 0.84, 0.81, 0.90, 0.87])
print("mean     :", round(scores.mean(), 4))
print("median   :", round(np.median(scores), 4))
print("variance :", round(scores.var(), 6), "(pop),", round(scores.var(ddof=1), 6), "(sample)")
print("stddev   :", round(scores.std(), 4))
assert np.isclose(scores.mean(), 0.843)
assert np.isclose(np.median(scores), 0.845)

mean     : 0.843
median   : 0.845
variance : 0.002121 (pop), 0.002357 (sample)
stddev   : 0.0461


> **1.2** A feature has mean 5.0 and std 1.2. Standardize the value 7.4 → z-score?

**Worked solution:** z = (x − μ)/σ = (7.4 − 5.0)/1.2 = 2.4/1.2 = **2.0** (2 std devs above the mean).

In [3]:
mu, sigma, x = 5.0, 1.2, 7.4
z = (x - mu) / sigma
print("z-score:", round(z, 3))
assert np.isclose(z, 2.0)

z-score: 2.0


> **1.3** Test scores are ~Normal(mean=70, std=8). A threshold of 85 is used to admit
> candidates. What fraction fall at or above it?

**Worked solution:** z = (85 − 70)/8 = 1.875. P(Z ≥ 1.875) = 1 − Φ(1.875) ≈ 1 − 0.9697 ≈ **0.0303 → ~3%**.

In [4]:
from scipy import stats
p = 1 - stats.norm.cdf(85, loc=70, scale=8)
print("P(X >= 85) =", round(p, 4))
assert np.isclose(p, 0.0304, atol=1e-3)

P(X >= 85) = 0.0304


## 2. Probability — uncertainty in ML

> **2.1** A classifier is 95% accurate before deployment. In production on 1,000 samples,
> expected number of correct predictions?

**Worked solution:** E[X] = n·p = 1000 × 0.95 = **950 correct**.

In [5]:
n, p = 1000, 0.95
print("expected correct:", n * p)
assert n * p == 950

expected correct: 950.0


> **2.2** Bayes' theorem. 1% of emails are spam. Spam detection flags 80% of spam and
> 5% of legitimate mail. A mail is flagged — what is P(spam | flagged)?

**Worked solution (Bayes):**
P(S|F) = [P(F|S)·P(S)] / [P(F|S)·P(S) + P(F|L)·P(L)]
= (0.80 × 0.01) / (0.80 × 0.01 + 0.05 × 0.99)
= 0.008 / 0.0575 ≈ **0.139**.

The intuition: false positives on the huge legitimate class dwarf the true positives,
so a flag alone is weak evidence — a classic example of *base-rate neglect*.

In [6]:
p_f_given_s, p_spam = 0.80, 0.01
p_f_given_l, p_legit = 0.05, 0.99
p_f = p_f_given_s * p_spam + p_f_given_l * p_legit
p_spam_given_f = p_f_given_s * p_spam / p_f
print("P(spam | flagged) =", round(p_spam_given_f, 3))
assert np.isclose(p_spam_given_f, 0.1391, atol=1e-3)

P(spam | flagged) = 0.139


> **2.3** Expected value in a recommender: reward of +10 for a click (p=0.15) and −1 otherwise.

**Worked solution:** E = 10(0.15) + (−1)(0.85) = 1.5 − 0.85 = **+0.65 per impression**.

In [7]:
e = 10 * 0.15 + (-1) * 0.85
print("expected reward:", e)
assert np.isclose(e, 0.65)

expected reward: 0.65


## 3. Linear algebra — the structure of features

> **3.1** Score = w₁·hours + w₂·gpa with w=(0.3, 0.7). Hours=(6, 9), GPA=(7.5, 8.0).
> Dot product = predictions (plus nothing else here).

**Worked solution:** w·x₁ = 0.3(6)+0.7(7.5) = 1.8+5.25 = **7.05**;
w·x₂ = 0.3(9)+0.7(8.0) = 2.7+5.6 = **8.30**.

In [8]:
w = np.array([0.3, 0.7])
X = np.array([[6, 7.5],
              [9, 8.0]])
pred = X @ w
print("predictions:", pred.round(3))
assert np.allclose(pred, [7.05, 8.30])

predictions: [7.05 8.3 ]


> **3.2** Solve a 2×2 system with matrix inverse: 3x + y = 9; x + 2y = 8.

**Worked solution:**
A = [[3, 1], [1, 2]], det(A) = 3·2 − 1·1 = 5.
A⁻¹ = (1/5)[[2, −1], [−1, 3]].
x = A⁻¹ b = (1/5)[[2,−1],[−1,3]]·[9,8] = (1/5)[18−8, −9+24] = [2, 3] → **x=2, y=3**.

In [9]:
A = np.array([[3.0, 1.0], [1.0, 2.0]])
b = np.array([9.0, 8.0])
print("det(A) =", round(np.linalg.det(A), 3))
x = np.linalg.solve(A, b)
print("solution:", x)
assert np.allclose(x, [2.0, 3.0])

det(A) = 5.0
solution: [2. 3.]


> **3.3** Eigenintuition for PCA. The covariance matrix C = [[2, 0.5], [0.5, 1]].
> Find its eigenvalues and explain what they mean for a PCA-style projection.

**Worked solution:** eigvalues of [[2, 0.5], [0.5, 1]] solve det(C − λI)=0:
(2−λ)(1−λ) − 0.25 = λ² − 3λ + 1.75 = 0 → **λ₁ ≈ 2.207**, **λ₂ ≈ 0.793**.
The largest eigenvalue is the variance captured along the first principal component;
λ₁/ (λ₁+λ₂) ≈ 73.6% of the variance lies on that axis.

In [10]:
C = np.array([[2.0, 0.5], [0.5, 1.0]])
w, v = np.linalg.eigh(C)
order = np.argsort(w)[::-1]
w, v = w[order], v[:, order]
print("eigenvalues:", w.round(3))
print("explained variance of PC1:", round(w[0] / w.sum(), 3))
assert np.allclose(w[0], 2.2071, atol=1e-2)

eigenvalues: [2.207 0.793]
explained variance of PC1: 0.736


## 4. Calculus — steepest descent & gradients

> **4.1** Squared-error loss for one sample: L(w) = (y − w·x)². For y=3, x=2, w=1,
> compute L and ∂L/∂w.

**Worked solution:** L = (3 − 1·2)² = 1² = **1**.
∂L/∂w = −2x(y − wx) = −2·2·(3 − 2) = **−4** (gradient points downhill in −L).

In [11]:
w, x, y = 1.0, 2.0, 3.0
L = (y - w * x) ** 2
dL = -2 * x * (y - w * x)
print("loss:", L, "| gradient dL/dw:", dL)
assert L == 1 and dL == -4

loss: 1.0 | gradient dL/dw: -4.0


> **4.2** Gradient descent: from w=0, x=2, y=6, lr=0.1, run 5 updates on
> L(w) = (y − wx)². Show w converging toward the optimum w* = 3.

**Worked solution:** update w ← w − lr·(−2x(y−wx)) = w + 2·lr·x(y−wx).
With lr=0.1, x=2 → step = 0.4·(6−2w). Trace:
w: 0 → 2.40 → 2.88 → 2.976 → 2.995 → 2.999 … approaching w* = y/x = 3.

In [12]:
w = 0.0; x, y = 2.0, 6.0; lr = 0.1
trail = [w]
for _ in range(5):
    grad = -2 * x * (y - w * x)
    w = w - lr * grad
    trail.append(round(w, 3))
print("w after each step:", trail)
print("optimum w* = y/x =", y / x)

w after each step: [0.0, 2.4, 2.88, 2.976, 2.995, 2.999]
optimum w* = y/x = 3.0


> **4.3** Numerical verification: finite-difference gradient of f(w) = w³ at w=2.

**Worked solution:** analytic f′(2) = 3·2² = 12. Central difference
[f(2+h) − f(2−h)]/2h with h=0.001 → ≈ **12.000001**; matches the analytic derivative.

In [13]:
def f(w): return w ** 3

h = 1e-6
fd = (f(2 + h) - f(2 - h)) / (2 * h)
print("analytic:", 3 * 2 ** 2, "| finite-diff:", round(fd, 6))
assert np.isclose(fd, 12.0, atol=1e-4)

analytic: 12 | finite-diff: 12.0


## Summary of worked solutions

- **Statistics:** mean/median/var/std and z-scores quantify feature spread; the 3%
  tail probability shows how threshold-based decisions rely on the normal model.
- **Probability:** Bayes' theorem + expected value are the foundation of naive Bayes
  classifiers and reward-based learning systems.
- **Linear algebra:** dot products are model predictions; inverses solve least-squares;
  eigenvalues tell us which directions carry the most variance (PCA).
- **Calculus:** the gradient is the downhill direction used by every gradient-descent
  optimizer; finite differences confirm the analytic result independently.